# Problem 6 - Danh gia shipper va doi tac van chuyen
Noi shipments, orders, order_items, reviews va returns de danh gia KPI, xu huong, chat luong dich vu va carrier.

In [ ]:
from pathlib import Path
from collections import Counter
import warnings
import unicodedata
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
SLA_DAYS = 4

def locate_repo():
    for start in [Path.cwd().resolve(), Path(r'C:/dpbngoc/DA & AI/DAAI_N2.3')]:
        for path in [start, *start.parents]:
            if (path / 'silver_data').exists(): return path
    raise FileNotFoundError('Khong tim thay silver_data')

REPO_ROOT = locate_repo()
def find_data_file(filename):
    paths = sorted((REPO_ROOT / 'silver_data').glob(f'.silver_pipeline_work_*/excel/{filename}'), reverse=True)
    for path in paths + [REPO_ROOT / filename]:
        if path.exists(): return path
    raise FileNotFoundError(filename)

def show_result(df, name, index=False):
    print(f'\n{"="*100}\n{name.replace("_"," " ).replace(".csv","").upper()}\n{"="*100}')
    display(df)

POSITIVE = ['chuyen nghiep','than thien','lich su','san sang ho tro','hai long','nhanh','tot','tuyet voi']
NEGATIVE = ['cham','tre','hu hong','that vong','khong hai long','khieu nai','thai do kem','khong ho tro']
COMPLAINT_GROUPS = {
    'Late delivery':['late','delay','cham','tre'], 'Damaged':['damage','broken','hu hong'],
    'Wrong item/size':['wrong','size','sai'], 'Quality':['quality','defect','chat luong'],
    'Service':['attitude','support','service','thai do','ho tro','dich vu']}

def sentiment(text):
    value = ''.join(c for c in unicodedata.normalize('NFD', str(text).lower()) if unicodedata.category(c) != 'Mn')
    pos = sum(w in value for w in POSITIVE); neg = sum(w in value for w in NEGATIVE)
    return 'Positive' if pos > neg else ('Negative' if neg > pos else 'Neutral')

def build_shipment_fact():
    shipments = pd.read_csv(find_data_file('shipments_realistic.csv'), low_memory=False)
    orders = pd.read_csv(find_data_file('orders_enriched.csv'), low_memory=False)
    items = pd.read_csv(find_data_file('order_items.csv'), low_memory=False)
    reviews = pd.read_csv(find_data_file('reviews.csv'), low_memory=False)
    returns = pd.read_csv(find_data_file('returns.csv'), low_memory=False)
    shipments['ship_date'] = pd.to_datetime(shipments['ship_date'], errors='coerce')
    shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'], errors='coerce')
    orders['order_status'] = orders['order_status'].astype(str).str.lower().str.strip()
    items[['quantity','unit_price','discount_amount']] = items[['quantity','unit_price','discount_amount']].apply(pd.to_numeric, errors='coerce')
    items['NetRevenue'] = (items['quantity'] * items['unit_price'] - items['discount_amount'].fillna(0)).clip(lower=0)
    order_value = items.groupby('order_id', as_index=False).agg(OrderValue=('NetRevenue','sum'))
    review_order = reviews.groupby('order_id', as_index=False).agg(ReviewRating=('rating','mean'), ReviewCount=('review_id','nunique'))
    return_order = returns.groupby('order_id', as_index=False).agg(ReturnFlag=('return_id', lambda s: 1), ReturnReason=('return_reason', lambda s: ' | '.join(sorted(set(s.dropna().astype(str))))), RefundAmount=('refund_amount','sum'))
    fact = shipments.merge(orders[['order_id','customer_id','order_status','comment']], on='order_id', how='left', validate='many_to_one')
    fact = fact.merge(order_value, on='order_id', how='left', validate='one_to_one').merge(review_order, on='order_id', how='left', validate='one_to_one').merge(return_order, on='order_id', how='left', validate='one_to_one')
    fact[['OrderValue','ReviewCount','ReturnFlag','RefundAmount']] = fact[['OrderValue','ReviewCount','ReturnFlag','RefundAmount']].fillna(0)
    fact['ReturnReason'] = fact['ReturnReason'].fillna('')
    fact['DeliveryDays'] = (fact['delivery_date'] - fact['ship_date']).dt.total_seconds() / 86400
    fact.loc[fact['DeliveryDays'] < 0, 'DeliveryDays'] = np.nan
    fact['IsSuccess'] = fact['order_status'].eq('delivered')
    fact['IsFailed'] = fact['order_status'].isin(['cancelled','created'])
    fact['IsReturn'] = fact['ReturnFlag'].eq(1) | fact['order_status'].eq('returned')
    fact['IsOnTime'] = fact['IsSuccess'] & fact['DeliveryDays'].le(SLA_DAYS)
    fact['IsLate'] = fact['IsSuccess'] & fact['DeliveryDays'].gt(SLA_DAYS)
    fact['Rating'] = fact['ReviewRating'].fillna(fact['shipper_rating'])
    fact['Sentiment'] = fact['comment'].fillna('').map(sentiment)
    complaint_text = (fact['comment'].fillna('') + ' ' + fact['ReturnReason']).map(lambda x: ''.join(c for c in unicodedata.normalize('NFD', str(x).lower()) if unicodedata.category(c) != 'Mn'))
    fact['IsComplaint'] = complaint_text.map(lambda x: any(w in x for w in NEGATIVE)) | fact['IsReturn']
    deliveries_per_customer = fact.groupby('customer_id')['order_id'].transform('nunique')
    fact['RepeatDeliveryCustomer'] = deliveries_per_customer.ge(2)
    fact['DeliveredValue'] = np.where(fact['IsSuccess'], fact['OrderValue'], 0)
    return fact

def calculate_shipper_kpis(fact):
    keys = ['shipper_id','shipper_name','shipper_company']
    kpi = fact.groupby(keys, as_index=False).agg(
        TotalOrders=('order_id','nunique'), OrdersDelivered=('IsSuccess','sum'),
        OrderValue=('DeliveredValue','sum'), Customers=('customer_id','nunique'),
        SuccessRate=('IsSuccess','mean'), AvgDeliveryDays=('DeliveryDays','mean'),
        LateRate=('IsLate','mean'), OnTimeRate=('IsOnTime','mean'),
        ReturnRate=('IsReturn','mean'), ComplaintRate=('IsComplaint','mean'),
        AvgRating=('Rating','mean'), RepeatDeliveryRate=('RepeatDeliveryCustomer','mean'))
    rate_cols = ['SuccessRate','LateRate','OnTimeRate','ReturnRate','ComplaintRate','RepeatDeliveryRate']
    kpi[rate_cols] *= 100
    kpi = kpi.rename(columns={col: col + '_%' for col in rate_cols})
    score_parts = {
        'RatingScore': kpi['AvgRating'].rank(pct=True),
        'SuccessScore': kpi['SuccessRate_%'].rank(pct=True),
        'SpeedScore': kpi['AvgDeliveryDays'].rank(pct=True, ascending=False),
        'ComplaintScore': kpi['ComplaintRate_%'].rank(pct=True, ascending=False),
        'RevenueScore': kpi['OrderValue'].rank(pct=True)}
    for name, values in score_parts.items(): kpi[name] = values
    kpi['KPI_Score'] = 100*(.25*kpi['RatingScore'] + .25*kpi['SuccessScore'] + .20*kpi['SpeedScore'] + .15*kpi['ComplaintScore'] + .15*kpi['RevenueScore'])
    kpi['PerformanceTier'] = pd.qcut(kpi['KPI_Score'].rank(method='first'), q=4, labels=['Underperformer','Average','High','Top'])
    kpi['OverallRank'] = kpi['KPI_Score'].rank(method='dense', ascending=False).astype(int)
    kpi['Recommendation'] = np.select([
        kpi['PerformanceTier'].astype(str).eq('Top'), kpi['SuccessRate_%'] < kpi['SuccessRate_%'].median(),
        kpi['AvgDeliveryDays'] > kpi['AvgDeliveryDays'].median(), kpi['ComplaintRate_%'] > kpi['ComplaintRate_%'].median()],
        ['Thuong va uu tien don gia tri cao; chia se best practice','Dao tao quy trinh giao thanh cong','Toi uu tuyen va ky nang quan ly thoi gian','Coaching chat luong dich vu va xu ly khieu nai'], default='Duy tri hieu suat va theo doi xu huong')
    show_result(kpi.sort_values('OverallRank'), 'shipper_kpi_ranking')
    return kpi.sort_values('OverallRank')

def track_trends_and_alerts(fact):
    work = fact.dropna(subset=['ship_date']).copy(); work['month'] = work['ship_date'].dt.to_period('M').dt.to_timestamp()
    monthly = work.groupby(['shipper_id','shipper_name','shipper_company','month'], as_index=False).agg(Orders=('order_id','nunique'), Revenue=('DeliveredValue','sum'), AvgRating=('Rating','mean'), SuccessRate=('IsSuccess','mean'), AvgDeliveryDays=('DeliveryDays','mean'))
    monthly['RevenueShare_%'] = monthly['Revenue'] / monthly.groupby('month')['Revenue'].transform('sum').replace(0,np.nan) * 100
    rows = []
    for shipper_id, group in monthly.groupby('shipper_id'):
        group = group.sort_values('month'); recent = group.tail(12).copy(); x = np.arange(len(recent))
        def slope(col): return float(np.polyfit(x, recent[col].fillna(recent[col].mean()).fillna(0), 1)[0]) if len(recent) > 1 else 0.0
        rev_mean = recent['Revenue'].mean(); rev_slope = slope('Revenue'); success_slope = slope('SuccessRate'); time_slope = slope('AvgDeliveryDays')
        share_mean = recent['RevenueShare_%'].mean(); share_slope = slope('RevenueShare_%'); norm_share_slope = share_slope / share_mean if share_mean else 0
        changes = recent['Revenue'].pct_change(); current_streak = 0
        for value in reversed(changes.iloc[1:].tolist()):
            if pd.notna(value) and value < 0: current_streak += 1
            else: break
        norm_rev_slope = rev_slope / rev_mean if rev_mean else 0
        revenue_cv = recent['Revenue'].std(ddof=1) / rev_mean if rev_mean else np.nan
        absolute_status = 'Cai thien' if norm_rev_slope >= .01 else ('Suy giam' if norm_rev_slope <= -.01 else 'On dinh')
        status = 'Cai thien' if norm_share_slope >= .01 and success_slope >= -.002 and time_slope <= .02 else ('Suy giam' if norm_share_slope <= -.01 or success_slope <= -.01 or (current_streak >= 4 and norm_share_slope < 0) else 'On dinh')
        rows.append({'shipper_id':shipper_id,'RevenueSlope12M':rev_slope,'NormalizedRevenueSlope12M':norm_rev_slope,'AbsoluteRevenueTrend':absolute_status,'RevenueShareSlope12M':share_slope,'NormalizedShareSlope12M':norm_share_slope,'RevenueCV12M':revenue_cv,'SuccessRateSlope12M':success_slope,'DeliveryTimeSlope12M':time_slope,'CurrentRevenueDeclineStreak':current_streak,'TrendStatus':status})
    trend = pd.DataFrame(rows)
    show_result(monthly, 'shipper_monthly_trends'); show_result(trend, 'shipper_trend_classification')
    return monthly, trend

def analyze_customer_feedback(fact):
    keys = ['shipper_id','shipper_name','shipper_company']
    quality = fact.groupby(keys, as_index=False).agg(AvgRating=('Rating','mean'), Complaints=('IsComplaint','sum'), FailedOrders=('IsFailed','sum'), OnTimeRate=('IsOnTime','mean'), RepeatDeliveryRate=('RepeatDeliveryCustomer','mean'), PositiveComments=('Sentiment', lambda s: (s=='Positive').sum()), NeutralComments=('Sentiment', lambda s: (s=='Neutral').sum()), NegativeComments=('Sentiment', lambda s: (s=='Negative').sum()))
    quality[['OnTimeRate','RepeatDeliveryRate']] *= 100
    quality = quality.rename(columns={'OnTimeRate':'OnTimeRate_%','RepeatDeliveryRate':'RepeatDeliveryRate_%'})
    text = (fact['comment'].fillna('') + ' ' + fact['ReturnReason'].fillna('')).map(lambda x: ''.join(c for c in unicodedata.normalize('NFD', str(x).lower()) if unicodedata.category(c) != 'Mn'))
    complaint_counts = []
    for group_name, words in COMPLAINT_GROUPS.items(): complaint_counts.append({'ComplaintCategory':group_name,'Mentions':int(text.map(lambda x: any(w in x for w in words)).sum())})
    complaint_counts = pd.DataFrame(complaint_counts).sort_values('Mentions', ascending=False)
    show_result(quality, 'shipper_service_quality'); show_result(complaint_counts, 'complaint_keyword_summary')
    try:
        from wordcloud import WordCloud
        cloud_text = ' '.join(text[fact['IsComplaint']].tolist()) or 'no complaint'
        image = WordCloud(width=1200, height=600, background_color='white').generate(cloud_text)
        plt.figure(figsize=(12,6)); plt.imshow(image, interpolation='bilinear'); plt.axis('off'); plt.tight_layout()
        plt.show()
    except Exception:
        plt.figure(figsize=(8,4)); plt.bar(complaint_counts['ComplaintCategory'], complaint_counts['Mentions'])
        plt.xticks(rotation=30, ha='right'); plt.title('Tan suat nhom khieu nai'); plt.tight_layout()
        plt.show()
    return quality, complaint_counts

def compare_carriers(fact):
    carrier = fact.groupby('shipper_company', as_index=False).agg(TotalOrders=('order_id','nunique'), Revenue=('DeliveredValue','sum'), AvgRating=('Rating','mean'), SuccessRate=('IsSuccess','mean'), AvgDeliveryDays=('DeliveryDays','mean'), ReturnRate=('IsReturn','mean'), ComplaintRate=('IsComplaint','mean'), OnTimeRate=('IsOnTime','mean'))
    rate_cols = ['SuccessRate','ReturnRate','ComplaintRate','OnTimeRate']
    carrier[rate_cols] *= 100
    carrier = carrier.rename(columns={col: col + '_%' for col in rate_cols})
    carrier['CarrierScore'] = 100*(.25*carrier['SuccessRate_%'].rank(pct=True) + .20*carrier['AvgRating'].rank(pct=True) + .20*carrier['AvgDeliveryDays'].rank(pct=True,ascending=False) + .15*carrier['ReturnRate_%'].rank(pct=True,ascending=False) + .10*carrier['ComplaintRate_%'].rank(pct=True,ascending=False) + .10*carrier['Revenue'].rank(pct=True))
    carrier['Rank'] = carrier['CarrierScore'].rank(method='dense', ascending=False).astype(int)
    carrier['Recommendation'] = np.where(carrier['Rank'].eq(1), 'Doi tac uu tien; dam phan nang cong suat va SLA', np.where(carrier['CarrierScore'] < carrier['CarrierScore'].median(), 'Yeu cau ke hoach cai thien SLA, return va complaint', 'Duy tri hop tac va theo doi KPI thang'))
    carrier = carrier.sort_values('Rank'); show_result(carrier, 'carrier_comparison')
    return carrier

def main():
    fact = build_shipment_fact(); show_result(pd.DataFrame([{'SLA_Days':SLA_DAYS,'OnTimeDefinition':f'Delivered and DeliveryDays <= {SLA_DAYS}','DataSource':'shipments_realistic + orders + order_items + reviews + returns'}]), 'sla_and_data_definition')
    kpi = calculate_shipper_kpis(fact); monthly, trend = track_trends_and_alerts(fact)
    quality, complaints = analyze_customer_feedback(fact); carrier = compare_carriers(fact)
    plt.figure(figsize=(10,6)); view = kpi.head(15).sort_values('KPI_Score')
    plt.barh(view['shipper_id'], view['KPI_Score']); plt.xlabel('KPI Score'); plt.title('Top 15 shipper')
    plt.tight_layout(); plt.show()
    plt.figure(figsize=(10,5)); plt.bar(carrier['shipper_company'], carrier['CarrierScore'])
    plt.xticks(rotation=30, ha='right'); plt.ylabel('Carrier Score'); plt.title('So sanh doi tac van chuyen')
    plt.tight_layout(); plt.show()
    top_ids = kpi.head(5)['shipper_id']
    plt.figure(figsize=(12,6))
    for shipper_id, group in monthly[monthly['shipper_id'].isin(top_ids)].groupby('shipper_id'):
        plt.plot(group['month'], group['Revenue'], label=shipper_id)
    plt.title('Xu huong doanh thu thang cua Top 5 shipper'); plt.ylabel('Revenue'); plt.legend(); plt.tight_layout(); plt.show()
    complaints.plot(kind='bar', x='ComplaintCategory', y='Mentions', figsize=(9,5), legend=False, title='Co cau khieu nai')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
    print('Hoan tat Problem 6 | Shipments:', len(fact), '| Shippers:', len(kpi), '| Carriers:', len(carrier))
    print('Tat ca bang va bieu do da hien thi truc tiep.')
    return kpi, monthly, trend, quality, complaints, carrier

shipper_kpi, shipper_monthly, shipper_trend, service_quality, complaint_summary, carrier_result = main()
